In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from ucimlrepo import fetch_ucirepo

import warnings
warnings.filterwarnings('ignore')

# Pobieranie zbioru Breast Cancer Wisconsin (Diagnostic) z UCI (ID: 17)
breast_cancer = fetch_ucirepo(id=17)

X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()

# Podział na zbiór treningowy (70%) i testowy (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Zbudujmy komitet. Modelem bazowym będzie bardzo płytkie drzewo decyzyjne (`max_depth=3`). Stworzymy komitet składający się z 10 takich drzew.

**Mierzone metryki:**
* **Accuracy (Dokładność):** Jaki procent pacjentów został zdiagnozowany poprawnie?
* **Czas predykcji (Inference Time):** Ile czasu zajmuje modelowi ocena całego zbioru testowego?
* **Pewność (Confidence):** Średnie prawdopodobieństwo wybranej klasy.

In [2]:
# 1. Definicja modelu
# estimator: algorytm bazowy (tutaj płytkie drzewo)
# n_estimators: liczba modeli w komitecie (tutaj 10 drzew)
base_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
bagging_model = BaggingClassifier(estimator=base_tree, n_estimators=10, random_state=42)

# 2. Trening modelu
bagging_model.fit(X_train, y_train)

# 3. Predykcja i pomiar czasu inferencji
inference_time = %timeit -o -q bagging_model.predict(X_test)

y_pred = bagging_model.predict(X_test)

# 4. Obliczenie dokładności (Accuracy)
accuracy = accuracy_score(y_test, y_pred)

# 5. Obliczenie pewności (Confidence)
# predict_proba zwraca prawdopodobieństwa dla każdej klasy (np. [0.2, 0.8] oznacza 80% pewności dla klasy 1)
# Bierzemy maksymalne prawdopodobieństwo dla każdej próbki i wyciągamy średnią
probabilities = bagging_model.predict_proba(X_test)
confidence = np.mean(np.max(probabilities, axis=1))

# 6. Wyświetlenie wyników
print(f"Dokładność (Acc) : {accuracy * 100:.2f}%")
print(f"Pewność (Conf)   : {confidence * 100:.2f}%")
print(f"Czas predykcji   : {inference_time}")

Dokładność (Acc) : 95.32%
Pewność (Conf)   : 94.85%
Czas predykcji   : 648 μs ± 2.76 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
